<div align="center">

# Week 09: Usability Testing: Observing People and Measuring with Confidence

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-09) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/Week09_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the reference coding of the seven pilot sessions of Part A and the age groups of the clinic's patients with their true completion rates. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


Z95 = 1.96   # the z value of a 95 % interval, as in the lecture

# The pilot of lab Part A: seven sessions of the current app, coded by the
# reference observer. Times run from "Please begin" to the end, in seconds.
REFERENCE = pd.DataFrame({
    "session": [1, 2, 3, 4, 5, 6, 7],
    "participant": ["Ayşe", "Hasan", "Elif", "Mehmet", "Zeynep", "Fatma", "Burak"],
    "outcome": ["completed", "gave up", "completed", "help", "completed",
                "help", "completed"],
    "errors": [0, 2, 1, 2, 0, 1, 0],
    "time_s": [51, 94, 68, 103, 46, 86, 59]})

# The patients of the clinic by age group (lab Part C) and their true
# completion rates with the current app
CLINIC_MIX = {"under 40": 0.25, "40 to 64": 0.40, "65 and over": 0.35}
TRUE_RATES = {"under 40": 0.90, "40 to 64": 0.75, "65 and over": 0.50}
print("Setup complete. Running in Colab:", IN_COLAB, "| pilot sessions:",
      len(REFERENCE), "| age groups:", list(CLINIC_MIX))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 9.1": (lambda: stats.norm.ppf(0.95), 1e-6),
    "Exercise 9.2": (lambda: stats.t.ppf(0.975, 12), 1e-6),
    "Exercise 9.3": (lambda: 0.85 ** 13, 1e-6),
    "Exercise A": (lambda: 11 / 25, 1e-6),
    "Exercise B": (lambda: adjusted_wald(3, 5)[2], 1e-6),
    "Exercise C": (lambda: math.exp(np.mean(np.log([60, 90, 240]))), 1e-6),
    "Exercise D": (lambda: math.ceil(1.96 ** 2 * 0.8 * 0.2 / 0.1 ** 2 - 1.96 ** 2), 1e-9),
    "Exercise E": (lambda: n1_test(45, 60, 54, 60)[1], 1e-6),
    "Exercise F": (lambda: geo_mean_ci(done_n["time_s"])[0], 1e-3),
}

## Python step 9: Distributions and tests with SciPy

Every number of a usability test comes from a few people, so it carries uncertainty. This step uses SciPy, the scientific library of Python, for the normal, the t and the binomial distribution and for a first statistical test &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#ref-1), [2](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#ref-2)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "TRUE_RATES" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-09/"
               "NB09_usability_testing.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### The normal distribution

In [ ]:
# Python practice: a small example for learning the language
print("P(Z < 1.96) =", round(stats.norm.cdf(1.96), 4))
print("z with 97.5 % below it:", round(stats.norm.ppf(0.975), 4))
inside = stats.norm.cdf(1.96) - stats.norm.cdf(-1.96)
print("share between -1.96 and 1.96:", round(inside, 4))

`stats.norm` is the standard normal distribution, with mean 0 and standard deviation 1. Its method `cdf` gives the probability of a value below a point, and `ppf` turns a probability back into the point. A 95 % interval leaves 2.5 % on each side, so it uses `ppf(0.975)`, which is 1.96.

### Small samples and the t distribution

In [ ]:
# Python practice: a small example for learning the language
for df in [3, 6, 12, 30, 1000]:
    print(f"t quantile for 95 %, {df:>4} degrees of freedom:",
          round(stats.t.ppf(0.975, df), 3))

With few participants, the standard deviation is itself uncertain. The t distribution of Student accounts for this with wider tails, and its *degrees of freedom* are usually $n - 1$. With 4 participants the quantile is 3.182 instead of 1.96, and with many it approaches 1.96.

### Counting successes with the binomial distribution

In [ ]:
# Python practice: a small example for learning the language
n, p = 7, 0.75
print("P(all 7 complete):", round(stats.binom.pmf(7, n, p), 4))
print("P(at most 4 complete):", round(stats.binom.cdf(4, n, p), 4))
rng = np.random.default_rng(RANDOM_STATE)
tests = rng.binomial(n, p, size=10000)          # 10 000 simulated tests
print("simulated share with at most 4:", round(float((tests <= 4).mean()), 4))

If each of $n$ participants completes a task with the probability $p$, the number of completions follows the binomial distribution. The method `pmf` gives the probability of one count, and `cdf` gives that of all counts up to it. `rng.binomial` draws simulated tests, and their shares approach the exact probabilities.

### A test gives a p-value

In [ ]:
# Python practice: a small example for learning the language
old = [61, 74, 58, 90, 66]          # times of five patients (s)
new = [52, 49, 63, 47, 55]
result = stats.ttest_ind(old, new, equal_var=False)
print("t =", round(result.statistic, 3), "| p =", round(result.pvalue, 3))

`stats.ttest_ind` compares the means of two independent groups. With `equal_var=False` it is Welch's t test, which the lecture uses. The *p-value* is the probability of a difference at least this large if both groups came from the same population. A small p-value speaks against that assumption.

**Quick check.** A test of 7 participants gives 5 completions. Why can the true completion rate still be 0.4 or 0.9?

<details><summary>Answer</summary>

Seven people are a small sample. Many true rates, from about 0.35 to 0.9, produce 5 of 7 with a reasonable probability, which is why a completion rate needs an interval.

</details>

### Exercises for Python step 9

**Exercise 9.1.** A 90 % interval leaves 5 % on each side. Store its z value in `z90`.

In [ ]:
z90 = None   # your answer

check("Exercise 9.1", z90)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
z90 = stats.norm.ppf(0.95)

check("Exercise 9.1", z90)

**Exercise 9.2.** Store the t quantile of a 95 % interval for 13 participants, that is 12 degrees of freedom, in `t12`.

In [ ]:
t12 = None   # your answer

check("Exercise 9.2", t12)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t12 = stats.t.ppf(0.975, 12)

check("Exercise 9.2", t12)

**Exercise 9.3.** Each of 13 patients completes a booking with the probability 0.85. Store the probability that all 13 complete in `p_all`.

In [ ]:
p_all = None   # your answer

check("Exercise 9.3", p_all)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_all = stats.binom.pmf(13, 13, 0.85)

check("Exercise 9.3", p_all)

## 1. Coding a test session: Your observation room &#91;[6](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#ref-6)&#93;

*Lecture: [The test session: Tasks, thinking aloud and coding](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#the-test-session)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "inside" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-09/"
               "NB09_usability_testing.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A usability test produces numbers only after an observer codes each session: Its outcome, its errors and its time. Python compares your codes of the interactive lab with the reference codes, with Cohen's kappa for the outcomes and with differences for the times.

**From the formulas to the code.** The observed agreement $p_o$ and the agreement expected by chance $p_e = \sum_k p_{a,k}\, p_{b,k}$ are computed inside `kappa(codes_a, codes_b)`, which returns $\kappa = (p_o - p_e)/(1 - p_e)$.

In [ ]:
def kappa(codes_a, codes_b):
    """Cohen's kappa of two observers who coded the same items."""
    a, b = pd.Series(list(codes_a)), pd.Series(list(codes_b))
    p_o = float((a.values == b.values).mean())
    share_a = a.value_counts(normalize=True)
    share_b = b.value_counts(normalize=True)
    p_e = float(sum(share_a.get(k, 0) * share_b.get(k, 0)
                    for k in set(a) | set(b)))
    return (p_o - p_e) / (1 - p_e) if p_e < 1 else 1.0

ref_codes = list(REFERENCE["outcome"])
slip = ref_codes.copy()
slip[5] = "completed"        # session 6, Fatma, coded as completed
print("worked example: kappa =", round(kappa(ref_codes, slip), 3))
print(pd.crosstab(pd.Series(ref_codes, name="reference"),
                  pd.Series(slip, name="observer")))

**Python note.** `value_counts(normalize=True)` gives the share of each code, such as 4/7 for completed. `pd.crosstab` counts how often each code of the reference met each code of the observer, so the agreements lie on its diagonal.

**Exercise A.** An observer codes session 2 as help instead of gave up and session 4 as completed instead of help. Store the kappa of this observer with the reference in `k_a`.

In [ ]:
k_a = None   # your answer

check("Exercise A", k_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
codes = ref_codes.copy()
codes[1], codes[3] = "help", "completed"
k_a = kappa(ref_codes, codes)

check("Exercise A", k_a)

Part A of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_SESSIONS`. Left empty, the variable makes the code use a simulated observer who presses the keys about one second late, misses an error and codes one session differently.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_SESSIONS = """
"""

def simulated_observer():
    """A typical first observer: late keys, one missed error, one slip."""
    rows = REFERENCE.copy()
    rows.loc[rows["session"] == 6, "outcome"] = "completed"
    rows.loc[rows["session"] == 3, "errors"] = 0
    rows["time_s"] = rows["time_s"] + [0.9, 1.1, 0.7, 1.3, 0.8, 1.2, 1.0]
    return rows[["session", "outcome", "errors", "time_s"]]

if MY_SESSIONS.strip():
    coded = pd.read_csv(io.StringIO(MY_SESSIONS.strip()))
    source = "your own coding from the interactive lab"
else:
    coded = simulated_observer()
    source = "a simulated observer"
both = REFERENCE.merge(coded[["session", "outcome", "errors", "time_s"]],
                       on="session", suffixes=("_ref", "_you"))
print("Data:", source)
agree = int((both["outcome_ref"] == both["outcome_you"]).sum())
print(f"outcomes: {agree} of {len(both)} agree, "
      f"kappa = {kappa(both['outcome_ref'], both['outcome_you']):.3f}")
gap = (both["time_s_you"] - both["time_s_ref"]).dropna()
print(f"times: yours minus the reference, mean {gap.mean():+.1f} s, "
      f"largest {gap.abs().max():.1f} s")
print("errors: reference", int(both["errors_ref"].sum()), "| yours",
      int(both["errors_you"].sum()))

**Python note.** `merge` joins your rows and the reference rows that share a session number, and `suffixes` names the two versions of each column. `dropna` removes sessions without a time, for example when the start key was never pressed.

In [ ]:
# Figure: your coding against the reference
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
x = both["session"]
ax[0].bar(x - 0.2, both["time_s_ref"], 0.4, color="#585E6E", label="reference")
ax[0].bar(x + 0.2, both["time_s_you"], 0.4, color="#5A4BB5", label="you")
ax[0].set_xlabel("session")
ax[0].set_ylabel("time (s)")
ax[0].legend()
codes = ["completed", "help", "gave up"]
table = pd.crosstab(both["outcome_ref"], both["outcome_you"]).reindex(
    index=codes, columns=codes, fill_value=0)
ax[1].imshow(table.values, cmap="Purples")
for i in range(3):
    for j in range(3):
        ax[1].text(j, i, int(table.values[i, j]), ha="center", va="center")
ax[1].set_xticks(range(3), codes)
ax[1].set_yticks(range(3), codes)
ax[1].set_xlabel("your code")
ax[1].set_ylabel("reference")
ax[1].grid(False)
plt.tight_layout()
plt.show()

**Python note.** The right panel is the cross table as an image. Agreements lie on the diagonal from the top left. Every number off the diagonal is a session that you and the reference coded differently.

**Observe and reflect.** Which session did you code differently from the reference, and which rule of the lecture decided it? How would two observers in your team agree on such rules before a test?

## 2. Completion rates with confidence intervals &#91;[9](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#ref-9)&#93;

*Lecture: [Completion rates with confidence](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#completion-rates)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "both" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-09/"
               "NB09_usability_testing.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A completion rate from seven or thirteen people is uncertain. Python computes the adjusted Wald interval of Agresti and Coull and checks with the binomial distribution how often such intervals contain the true rate.

**From the formulas to the code.** The adjusted proportion $\hat{p}_{adj} = (x + z^2/2)/(n + z^2)$ and the interval $\hat{p}_{adj} \pm z\sqrt{\hat{p}_{adj}(1 - \hat{p}_{adj})/(n + z^2)}$ are `adjusted_wald(x, n)`. The plain interval $\hat{p} \pm z\sqrt{\hat{p}(1 - \hat{p})/n}$ is `wald(x, n)`.

In [ ]:
def adjusted_wald(x, n, z=Z95):
    """Adjusted Wald interval of a completion rate: (p_adj, low, high)."""
    p_adj = (x + z ** 2 / 2) / (n + z ** 2)
    margin = z * math.sqrt(p_adj * (1 - p_adj) / (n + z ** 2))
    return p_adj, max(0.0, p_adj - margin), min(1.0, p_adj + margin)

def wald(x, n, z=Z95):
    """The plain Wald interval: (p, low, high)."""
    p = x / n
    margin = z * math.sqrt(p * (1 - p) / n)
    return p, max(0.0, p - margin), min(1.0, p + margin)

for x, n in [(4, 7), (7, 7)]:
    p_adj, low, high = adjusted_wald(x, n)
    p, wl, wh = wald(x, n)
    print(f"{x} of {n}: adjusted {p_adj:.3f}, {low:.3f} to {high:.3f}"
          f" | plain Wald {p:.3f}, {wl:.3f} to {wh:.3f}")
done = both["outcome_you"] == "completed"
p_adj, low, high = adjusted_wald(int(done.sum()), len(both))
print(f"your coding of Part A: {int(done.sum())} of {len(both)} completed "
      f"without help, interval {low:.3f} to {high:.3f}")

**Python note.** `max(0.0, ...)` and `min(1.0, ...)` keep the interval between 0 and 1. Assisted sessions count as not completed, so only the code completed enters `done`.

**Exercise B.** Three of five participants complete a task. Compute the upper end of the adjusted Wald interval by hand first, then with `adjusted_wald`, and store it in `upper`.

In [ ]:
upper = None   # your answer

check("Exercise B", upper)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
upper = adjusted_wald(3, 5)[2]

check("Exercise B", upper)

In [ ]:
def coverage(p, n, method):
    """Probability that the interval of a test with n people contains p."""
    total = 0.0
    for x in range(n + 1):
        _, low, high = method(x, n)
        if low <= p <= high:
            total += stats.binom.pmf(x, n, p)
    return total

grid = np.linspace(0.05, 0.95, 181)
for n in (7, 13):
    cw = np.array([coverage(p, n, wald) for p in grid])
    ca = np.array([coverage(p, n, adjusted_wald) for p in grid])
    print(f"n = {n:>2}: mean coverage, plain Wald {cw.mean():.3f}, "
          f"adjusted Wald {ca.mean():.3f}")
print("n = 12, p = 0.75: plain Wald", round(coverage(0.75, 12, wald), 3),
      "| adjusted", round(coverage(0.75, 12, adjusted_wald), 3))

**Python note.** `coverage` adds the binomial probabilities of all results whose interval contains the true rate, so it gives the exact share of tests in which the interval is right. A 95 % interval should reach about 0.95.

In [ ]:
# Figure: how often the intervals contain the true rate
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for a, n in zip(ax, (7, 13)):
    a.plot(grid, [coverage(p, n, wald) for p in grid], color="#B53F2E",
           label="plain Wald")
    a.plot(grid, [coverage(p, n, adjusted_wald) for p in grid],
           color="#5A4BB5", label="adjusted Wald")
    a.axhline(0.95, ls=":", color="#585E6E")
    a.set_title(f"tests with {n} participants")
    a.set_xlabel("true completion rate")
ax[0].set_ylabel("coverage")
ax[0].set_ylim(0.4, 1.0)
ax[0].legend()
plt.tight_layout()
plt.show()

**Python note.** The coverage jumps because only whole numbers of completions are possible. The plain interval drops far below 0.95 near the high rates that good designs reach, while the adjusted interval stays close to it.

**Explore.** Move the sliders: The panel draws both intervals for $x$ completions of $n$ participants. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(x=4, n=7):
    x = min(x, n)
    rows = [("adjusted Wald", adjusted_wald(x, n), "#5A4BB5"),
            ("plain Wald", wald(x, n), "#B53F2E")]
    plt.figure(figsize=(6, 1.8))
    for k, (name, (p, low, high), colour) in enumerate(rows):
        plt.plot([low, high], [k, k], color=colour, lw=6)
        plt.plot(p, k, "o", color="#1E2433")
        print(f"{name}: {low:.3f} to {high:.3f}")
    plt.yticks([0, 1], [r[0] for r in rows])
    plt.xlim(0, 1)
    plt.show()

widgets.interact(explore, x=widgets.IntSlider(value=4, min=0, max=30),
                 n=widgets.IntSlider(value=7, min=1, max=30));

**Observe and reflect.** With how many participants does the adjusted interval of your Part A coding become narrower than 0.2 on each side, if the completion rate stays the same? Try it with the sliders.

## 3. Time on task: Logarithms and the geometric mean &#91;[10](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#ref-10)&#93;

*Lecture: [Time on task: Skewed times and the geometric mean](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#time-on-task)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "coverage" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-09/"
               "NB09_usability_testing.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Task times are skewed: Most people are quick, and a few take much longer. Python averages the logarithms of the times, turns the result back into seconds and compares two designs with Welch's t test on the logarithms.

**From the formulas to the code.** The geometric mean $e^{\bar{y}}$ with $y_i = \ln t_i$ and its interval $e^{\bar{y} \pm t_{n-1}\, s_y/\sqrt{n}}$ are `geo_mean_ci(times)`. Welch's statistic $t = (\bar{y}_A - \bar{y}_B)/\sqrt{s_A^2/n_A + s_B^2/n_B}$ is `stats.ttest_ind(..., equal_var=False)` on the logarithms.

In [ ]:
def geo_mean_ci(times, conf=0.95):
    """Geometric mean of task times with its t interval from the logarithms."""
    logs = np.log(np.asarray(times, dtype=float))
    n, mean, sd = len(logs), logs.mean(), logs.std(ddof=1)
    t = stats.t.ppf(0.5 + conf / 2, n - 1)
    margin = t * sd / math.sqrt(n)
    return math.exp(mean), math.exp(mean - margin), math.exp(mean + margin)

pilot = [51, 68, 46, 59]            # completed sessions of Part A (s)
redesign_pilot = [31, 39, 43, 49]   # a pilot of the redesign (s)
for name, times in [("current", pilot), ("redesign", redesign_pilot)]:
    gm, low, high = geo_mean_ci(times)
    print(f"{name}: geometric mean {gm:.1f} s, interval {low:.1f} to "
          f"{high:.1f} s, arithmetic mean {np.mean(times):.1f} s")
res = stats.ttest_ind(np.log(pilot), np.log(redesign_pilot),
                      equal_var=False)
print(f"Welch on the logarithms: t = {res.statistic:.3f}, "
      f"p = {res.pvalue:.3f}")

**Python note.** `logs.std(ddof=1)` divides by $n - 1$, as the sample standard deviation of the lecture. `stats.t.ppf(0.975, 3)` is 3.182 for four times.

**Exercise C.** Three patients take 60, 90 and 240 s. Store their geometric mean in `gm_c` and compare it with their arithmetic mean of 130 s.

In [ ]:
gm_c = None   # your answer

check("Exercise C", gm_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
gm_c = math.exp(np.mean(np.log([60, 90, 240])))

check("Exercise C", gm_c)

In [ ]:
yours = both.loc[both["outcome_you"] == "completed", "time_s_you"].dropna()
if len(yours) >= 2:
    gm, low, high = geo_mean_ci(yours)
    print(f"your completed sessions: {len(yours)}, geometric mean {gm:.1f} s,"
          f" interval {low:.1f} to {high:.1f} s")
else:
    print("Fewer than two completed sessions in your coding.")

def estimator_errors(n, reps=4000, median=60, sigma=0.5, seed=RANDOM_STATE):
    """Mean absolute error of three averages of n log-normal task times."""
    rng = np.random.default_rng(seed)
    t = rng.lognormal(math.log(median), sigma, size=(reps, n))
    est = {"arithmetic mean": t.mean(axis=1),
           "median": np.median(t, axis=1),
           "geometric mean": np.exp(np.log(t).mean(axis=1))}
    return {k: float(np.abs(v - median).mean()) for k, v in est.items()}

for n in (5, 10, 20):
    errs = estimator_errors(n)
    print(f"n = {n:>2}: " + ", ".join(f"{k} {v:.1f} s" for k, v in errs.items()))

**Python note.** `rng.lognormal` draws times whose logarithms are normal, the usual model of task times. The true centre of these times, their median, is 60 s, and each average is compared with it.

In [ ]:
# Figure: skewed task times and the three averages
rng = np.random.default_rng(RANDOM_STATE)
many = rng.lognormal(math.log(60), 0.5, size=3000)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(many, bins=60, color="#C9C3E6")
for value, name, colour in [(many.mean(), "arithmetic mean", "#B53F2E"),
                            (np.median(many), "median", "#2E8B57"),
                            (math.exp(np.log(many).mean()), "geometric mean",
                             "#5A4BB5")]:
    ax[0].axvline(value, color=colour, lw=2, label=f"{name} {value:.0f} s")
ax[0].set_xlabel("task time (s)")
ax[0].legend(fontsize=8)
ns = [3, 5, 8, 12, 20, 30]
table = pd.DataFrame([estimator_errors(n) for n in ns], index=ns)
for name, colour in [("arithmetic mean", "#B53F2E"), ("median", "#2E8B57"),
                     ("geometric mean", "#5A4BB5")]:
    ax[1].plot(ns, table[name], "o-", color=colour, label=name)
ax[1].set_xlabel("participants")
ax[1].set_ylabel("mean absolute error (s)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** The long right tail pulls the arithmetic mean above the median. The right panel repeats 4000 simulated tests for each sample size and shows how far each average lands from the true centre.

**Observe and reflect.** Which average of your completed sessions would you report to the clinic, and why? What would one patient who needed five minutes do to each of the three averages?

## 4. Recruiting: Your participants and the patients &#91;[12](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#ref-12)&#93;

*Lecture: [Who takes part, and how many](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#recruiting)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "estimator_errors" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-09/"
               "NB09_usability_testing.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A test speaks for the patients only if its participants resemble them. Python compares the age groups of your recruits with the patients of the clinic and weights the completion rates by the patient mix. It also computes how many participants an interval of a given width needs.

**From the formulas to the code.** The weighted rate $\hat{p}_w = \sum_g w_g\, \hat{p}_g$ is `weighted_rate(results)`, and $n = z^2 p(1 - p)/d^2 - z^2$, rounded up, is `sample_size(p, d)`.

In [ ]:
def weighted_rate(results, mix=CLINIC_MIX):
    """Completion rate weighted by the age groups of the clinic's patients."""
    rates = results.groupby("group")["completed"].mean()
    present = [g for g in mix if g in rates.index]
    w = np.array([mix[g] for g in present])
    return float((w * rates[present].values).sum() / w.sum()), rates

def sample_size(p, margin, z=Z95):
    """Participants for an adjusted Wald interval of +/- margin around p."""
    return math.ceil(z ** 2 * p * (1 - p) / margin ** 2 - z ** 2)

example = pd.DataFrame({"group": ["under 40"] * 6 + ["40 to 64", "65 and over"],
                        "completed": [1, 1, 1, 1, 1, 1, 1, 0]})
pw, rates = weighted_rate(example)
print(f"worked example: simple {example['completed'].mean():.3f}, "
      f"weighted {pw:.3f}")
truth = sum(CLINIC_MIX[g] * TRUE_RATES[g] for g in CLINIC_MIX)
print(f"true rate of the clinic's patients: {truth:.3f}")
print("participants for +/- 0.10:", sample_size(0.7, 0.10),
      "| for +/- 0.20:", sample_size(0.7, 0.20))

**Python note.** `groupby("group")["completed"].mean()` gives the completion rate of each age group. A group without participants cannot be weighted, so `present` keeps only the groups found and rescales their weights.

**Exercise D.** The team expects a completion rate of about 0.8 and wants an interval of $\pm 0.10$. Store the number of participants in `n_d`.

In [ ]:
n_d = None   # your answer

check("Exercise D", n_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_d = sample_size(0.8, 0.10)

check("Exercise D", n_d)

Part C of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_RECRUITS`. Left empty, the variable makes the code use a student who invites everyone in round 1 and fills the quotas in round 2.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_RECRUITS = """
"""

DEFAULT_RECRUITS = """
round,arrival,name,age,group,invited,agreed,completed
1,1,Deniz,21,under 40,1,1,1
1,2,Ece,20,under 40,1,1,1
1,3,Can,22,under 40,1,1,1
1,4,Mert,23,under 40,1,1,1
1,5,Selin,19,under 40,1,1,1
1,6,Hasan,71,65 and over,1,0,
1,7,Kerem,24,under 40,1,1,1
1,8,Fatma,68,65 and over,1,1,0
1,9,Murat,52,40 to 64,1,1,1
2,1,Arda,25,under 40,1,1,1
2,2,Kemal,67,65 and over,1,0,
2,3,Sibel,48,40 to 64,1,1,1
2,4,Yusuf,22,under 40,1,1,1
2,5,Elif,30,under 40,0,0,
2,6,Ahmet,72,65 and over,1,1,0
2,7,Hülya,56,40 to 64,1,1,1
2,8,Cem,27,under 40,0,0,
2,9,Necla,69,65 and over,1,1,1
2,10,Rıza,63,40 to 64,1,0,
2,11,Berk,20,under 40,0,0,
2,12,Saadet,79,65 and over,1,0,
2,13,Tuncay,44,40 to 64,1,1,0
2,14,Meltem,35,under 40,0,0,
2,15,Şükrü,75,65 and over,1,1,0
"""
text = MY_RECRUITS.strip() or DEFAULT_RECRUITS.strip()
source = "your own recruiting" if MY_RECRUITS.strip() else "a typical student"
recruits = pd.read_csv(io.StringIO(text))
tested = recruits[recruits["agreed"] == 1]
print("Data:", source)
for rnd, part in tested.groupby("round"):
    x, n = int(part["completed"].sum()), len(part)
    _, low, high = adjusted_wald(x, n)
    pw, _ = weighted_rate(part)
    mix = part["group"].value_counts().reindex(list(CLINIC_MIX), fill_value=0)
    print(f"round {rnd}: {x} of {n} completed, interval {low:.2f} to "
          f"{high:.2f}, weighted {pw:.3f} | ages: " +
          ", ".join(f"{g} {c}" for g, c in mix.items()))
invited = recruits[recruits["invited"] == 1]
asked = invited.groupby("group")["agreed"].mean().reindex(list(CLINIC_MIX))
print("share who agreed when invited:", asked.round(2).to_dict())

**Python note.** `recruits["agreed"] == 1` keeps the people who took the test. `reindex` lists every age group in the order of the clinic, also a group with no participant.

In [ ]:
def recruiting_estimates(kind, n=8, reps=4000, seed=RANDOM_STATE):
    """Simple completion rates of many simulated samples of n participants."""
    rng = np.random.default_rng(seed)
    groups = list(CLINIC_MIX)
    if kind == "quota":
        counts = np.tile([2, 3, 3], (reps, 1))
    else:
        shares = [0.60, 0.25, 0.15] if kind == "entrance" else \
            list(CLINIC_MIX.values())
        counts = rng.multinomial(n, shares, size=reps)
    done = sum(rng.binomial(counts[:, k], TRUE_RATES[g])
               for k, g in enumerate(groups))
    return done / n

fig, ax = plt.subplots(figsize=(7, 3.2))
for k, (kind, colour) in enumerate([("entrance", "#B53F2E"),
                                    ("random", "#585E6E"),
                                    ("quota", "#5A4BB5")]):
    est = recruiting_estimates(kind)
    print(f"{kind:>8}: mean {est.mean():.3f}, spread (SD) {est.std():.3f}")
    ax.boxplot(est, positions=[k], widths=0.5, vert=False,
               patch_artist=True, boxprops=dict(facecolor=colour, alpha=0.4))
ax.axvline(truth, ls=":", color="#1E2433", label=f"true rate {truth:.2f}")
ax.set_yticks([0, 1, 2], ["first at the entrance", "random patients",
                          "quotas 2 + 3 + 3"])
ax.set_xlabel("completion rate of a test with 8 participants")
ax.legend()
plt.tight_layout()
plt.show()

**Python note.** `rng.multinomial` splits eight participants into the three age groups with the given shares, once for each simulated test. The shares 0.60, 0.25 and 0.15 describe an entrance where young people agree first, as in round 1 of Part C.

**Observe and reflect.** How far was your first round from the true rate of 0.70, and how far your second? Which patients of the clinic would still be missing from a test that fills the quotas at the entrance?

## 5. Comparing two designs: Your A/B tests &#91;[13](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#ref-13)&#93;

*Lecture: [Comparing two designs: Tests, power and peeking](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#comparing-designs)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "recruiting_estimates" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-09/"
               "NB09_usability_testing.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** An A/B test compares two designs with different people. Python computes the N-1 two-proportion test and the number of patients per design that a difference needs. It also shows what happens to false alarms when a team looks at the results again and again.

**From the formulas to the code.** The statistic $z = (\hat{p}_B - \hat{p}_A)/\sqrt{\bar{p}(1 - \bar{p})(1/n_A + 1/n_B)} \cdot \sqrt{(N - 1)/N}$ and its p-value are `n1_test(xa, na, xb, nb)`, and the patients per design are `per_group(pa, pb)`.

In [ ]:
def n1_test(xa, na, xb, nb):
    """N-1 two-proportion test: z and the two-sided p-value."""
    N = na + nb
    pbar = (xa + xb) / N
    if pbar in (0, 1):
        return 0.0, 1.0
    se = math.sqrt(pbar * (1 - pbar) * (1 / na + 1 / nb))
    z = (xb / nb - xa / na) / se * math.sqrt((N - 1) / N)
    return z, 2 * stats.norm.sf(abs(z))

def per_group(pa, pb, z_alpha=Z95, power=0.8):
    """Patients per design for a two-sided test at 5 % with the given power."""
    z_beta = stats.norm.ppf(power)
    pbar = (pa + pb) / 2
    top = (z_alpha * math.sqrt(2 * pbar * (1 - pbar))
           + z_beta * math.sqrt(pa * (1 - pa) + pb * (1 - pb)))
    return math.ceil(top ** 2 / (pa - pb) ** 2)

z, p = n1_test(70, 100, 80, 100)
print(f"worked example: z = {z:.3f}, p = {p:.3f}")
print("patients per kiosk for 0.70 against 0.80:", per_group(0.70, 0.80))

**Python note.** `stats.norm.sf(abs(z))` is the probability above $|z|$, and doubling it counts both directions, so the test notices a difference in favour of either design.

**Exercise E.** In an A/B test, 45 of 60 patients complete the booking at kiosk A and 54 of 60 at kiosk B. Store the p-value of the N-1 test in `p_e`.

In [ ]:
p_e = None   # your answer

check("Exercise E", p_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_e = n1_test(45, 60, 54, 60)[1]

check("Exercise E", p_e)

Part D of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_AB`. Left empty, the variable makes the code use a student who stops each test as soon as p falls below 0.05.

In [ ]:
# Paste the results of lab Part D between the triple quotes.
MY_AB = """
"""

DEFAULT_AB = """
round,looks,per_kiosk,completed_a,completed_b,p_value,decision,true_a,true_b,verdict
1,2,20,18,12,0.031,A better,0.75,0.75,false alarm
2,5,50,38,46,0.030,B better,0.7,0.85,right
3,10,100,79,83,0.472,no difference,0.8,0.8,right
4,10,100,70,74,0.530,no difference,0.72,0.78,missed
5,7,70,46,58,0.021,B better,0.65,0.8,right
6,10,100,79,75,0.503,no difference,0.78,0.78,right
"""
text = MY_AB.strip() or DEFAULT_AB.strip()
source = "your own decisions" if MY_AB.strip() else "a student who stops early"
ab = pd.read_csv(io.StringIO(text))
print("Data:", source)
print(ab[["round", "looks", "completed_a", "completed_b", "p_value",
          "decision", "verdict"]].to_string(index=False))
print("verdicts:", ab["verdict"].value_counts().to_dict())

**Python note.** `to_string(index=False)` prints the table without its row numbers. The verdict compares your decision with the true rates of each round, which the interactive lab revealed after the decision.

In [ ]:
def aa_tests(p=0.75, looks=10, step=10, reps=4000, seed=RANDOM_STATE):
    """p-values of simulated A/A tests after every look of 'step' patients."""
    rng = np.random.default_rng(seed)
    a = rng.binomial(step, p, size=(reps, looks)).cumsum(axis=1)
    b = rng.binomial(step, p, size=(reps, looks)).cumsum(axis=1)
    n = step * np.arange(1, looks + 1)
    pbar = (a + b) / (2 * n)
    se = np.sqrt(pbar * (1 - pbar) * 2 / n)
    with np.errstate(divide="ignore", invalid="ignore"):
        z = (b - a) / n / se * np.sqrt((2 * n - 1) / (2 * n))
    z = np.nan_to_num(z)
    return 2 * stats.norm.sf(np.abs(z))

pv = aa_tests()
first_alarm = (pv < 0.05).cumsum(axis=1) > 0
print(f"A/A tests with p < 0.05 at the planned end: "
      f"{(pv[:, -1] < 0.05).mean():.3f}")
print(f"A/A tests with p < 0.05 at one of ten looks: "
      f"{first_alarm[:, -1].mean():.3f}")
rng = np.random.default_rng(RANDOM_STATE)
for n in (100, per_group(0.70, 0.80)):
    xa = rng.binomial(n, 0.70, 4000)
    xb = rng.binomial(n, 0.80, 4000)
    found = np.mean([n1_test(a, n, b, n)[1] < 0.05 for a, b in zip(xa, xb)])
    print(f"0.70 against 0.80 with {n} per kiosk: found in {found:.2f} of tests")

**Python note.** `cumsum(axis=1)` adds the completions look by look. `(pv < 0.05).cumsum(axis=1) > 0` marks every look at or after the first p-value below 0.05, the moment a team that peeks would stop.

In [ ]:
# Figure: wandering p-values and the price of peeking
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
looks = np.arange(1, 11)
for row in pv[:20]:
    ax[0].plot(looks, row, color="#5A4BB5", alpha=0.35)
ax[0].axhline(0.05, color="#B53F2E", ls=":")
ax[0].set_yscale("log")
ax[0].set_xlabel("look, 10 more patients per kiosk each")
ax[0].set_ylabel("p-value (log scale)")
ax[0].set_title("Twenty A/A tests")
ax[1].plot(looks, first_alarm.mean(axis=0), "o-", color="#B53F2E",
           label="stop at the first p < 0.05")
ax[1].plot(looks, (pv < 0.05).mean(axis=0), "o-", color="#585E6E",
           label="decide only at this look")
ax[1].set_xlabel("number of looks")
ax[1].set_ylabel("share of false alarms")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** Both kiosks of an A/A test are equal, so every p-value below 0.05 is a false alarm. The grey line stays near 0.05, while the red line grows with every extra look.

**Observe and reflect.** In which round of Part D did you stop too early or too late? Which rule would you write into the plan of the clinic's A/B test before it starts?

## 6. The clinic app, stage 9 &#91;[15](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#ref-15)&#93;

*Lecture: [The clinic app, stage 9: A usability test of both designs](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#the-clinic-app-stage-9)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "aa_tests" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-09/"
               "NB09_usability_testing.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 9 tests the current app and the redesign of Week 8 with 13 patients each. Python computes the completion rates with their intervals, the geometric mean times and the errors. It adds the scores of the System Usability Scale (SUS) and the first impressions, compares the designs and asks how many patients a clear answer would need.

In [ ]:
PATIENTS = pd.DataFrame({
    "design": ["current"] * 13 + ["redesign"] * 13,
    "group": (["under 40"] * 3 + ["40 to 64"] * 5 + ["65 and over"] * 5) * 2,
    "outcome": ["completed"] * 6 + ["help", "gave up"]
               + ["completed"] * 2 + ["gave up", "help", "gave up"]
               + ["completed"] * 11 + ["help", "gave up"],
    "time_s": [58, 66, 61, 74, 83, 97, 166, 182, 121, 140, 175, 204, 190,
               36, 41, 39, 44, 47, 52, 49, 58, 63, 71, 82, 151, 139],
    "errors": [0, 1, 0, 1, 2, 1, 3, 4, 2, 3, 4, 3, 5,
               0, 0, 0, 0, 1, 0, 0, 1, 1, 2, 1, 2, 3],
    "sus": [70, 67.5, 75, 62.5, 57.5, 65, 45, 40, 52.5, 50, 37.5, 42.5, 35,
            85, 90, 82.5, 80, 77.5, 85, 75, 72.5, 77.5, 70, 72.5, 60, 57.5],
    "appeal": [5, 4, 5, 4, 5, 4, 3, 4, 4, 5, 3, 4, 4,
               7, 8, 7, 6, 7, 7, 6, 7, 6, 7, 6, 5, 6]})

STAGE9 = {}
for name, part in PATIENTS.groupby("design", sort=False):
    ok = part[part["outcome"] == "completed"]
    x, n = len(ok), len(part)
    _, low, high = adjusted_wald(x, n)
    gm, glow, ghigh = geo_mean_ci(ok["time_s"])
    sus = part["sus"]
    half = stats.t.ppf(0.975, n - 1) * sus.std(ddof=1) / math.sqrt(n)
    STAGE9[f"completed without help, {name}"] = f"{x} of {n}"
    STAGE9[f"interval of the completion rate, {name}"] = f"{low:.3f} to {high:.3f}"
    STAGE9[f"geometric mean time, {name} (s)"] = round(gm, 1)
    STAGE9[f"interval of the time, {name} (s)"] = f"{glow:.1f} to {ghigh:.1f}"
    STAGE9[f"errors per patient, {name}"] = round(part["errors"].mean(), 2)
    STAGE9[f"SUS score, {name}"] = round(sus.mean(), 1)
    STAGE9[f"interval of the SUS score, {name}"] = (
        f"{sus.mean() - half:.1f} to {sus.mean() + half:.1f}")
    STAGE9[f"first impression, {name} (1 to 9)"] = round(part["appeal"].mean(), 2)

cur = PATIENTS[PATIENTS["design"] == "current"]
new = PATIENTS[PATIENTS["design"] == "redesign"]
done_c = cur[cur["outcome"] == "completed"]
done_n = new[new["outcome"] == "completed"]
STAGE9["p of the completion rates (N-1 test)"] = round(
    n1_test(len(done_c), 13, len(done_n), 13)[1], 3)
STAGE9["p of the times (Welch on logarithms)"] = round(stats.ttest_ind(
    np.log(done_c["time_s"]), np.log(done_n["time_s"]), equal_var=False).pvalue, 4)
STAGE9["p of the SUS scores (Welch)"] = round(stats.ttest_ind(
    cur["sus"], new["sus"], equal_var=False).pvalue, 5)
STAGE9["p of the first impressions (Welch)"] = round(stats.ttest_ind(
    cur["appeal"], new["appeal"], equal_var=False).pvalue, 5)
STAGE9["patients per design for these completion rates"] = per_group(
    len(done_c) / 13, len(done_n) / 13)
for key, value in STAGE9.items():
    print(f"{key}: {value}")
print(pd.crosstab([PATIENTS["design"], PATIENTS["group"]],
                  PATIENTS["outcome"]))

**Python note.** The table holds one row per patient, 13 for each design, with three patients under 40, five aged 40 to 64 and five aged 65 and over. `groupby("design", sort=False)` keeps the order of the table, so the current app comes first.

In [ ]:
# Figure: the usability test of stage 9
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
colours = {"current": "#B53F2E", "redesign": "#2E8B57"}
for k, (name, part) in enumerate(PATIENTS.groupby("design", sort=False)):
    ok = part[part["outcome"] == "completed"]
    rate = len(ok) / len(part)
    _, low, high = adjusted_wald(len(ok), len(part))
    ax[0].errorbar(k, rate, yerr=[[rate - low], [high - rate]], fmt="o",
                   capsize=8, color=colours[name], lw=2)
    gm, glow, ghigh = geo_mean_ci(ok["time_s"])
    ax[1].plot(np.full(len(ok), k) + np.linspace(-0.12, 0.12, len(ok)),
               ok["time_s"], "o", color=colours[name], alpha=0.6)
    ax[1].errorbar(k + 0.3, gm, yerr=[[gm - glow], [ghigh - gm]], fmt="s",
                   capsize=6, color="#1E2433")
    ax[2].plot(np.full(len(part), k) + np.linspace(-0.12, 0.12, len(part)),
               part["sus"], "o", color=colours[name], alpha=0.6)
    ax[2].plot(k + 0.3, part["sus"].mean(), "s", color="#1E2433")
for a in ax:
    a.set_xticks([0, 1], ["current", "redesign"])
    a.set_xlim(-0.5, 1.6)
ax[0].set_ylim(0, 1)
ax[0].set_title("completed without help")
ax[1].set_yscale("log")
ax[1].set_title("time of the completed bookings (s)")
ax[2].axhline(68, ls=":", color="#585E6E")
ax[2].set_title("SUS score; dotted: average of 68")
plt.tight_layout()
plt.show()

**Python note.** Squares mark the geometric mean with its interval and the mean SUS score. The time axis is logarithmic, so equal ratios of times have equal distances.

**Exercise F.** Store the geometric mean time of the completed bookings with the redesign, in seconds, in `gm_f`.

In [ ]:
gm_f = None   # your answer

check("Exercise F", gm_f)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
gm_f = geo_mean_ci(done_n["time_s"])[0]

check("Exercise F", gm_f)

**Observe and reflect.** Which result of stage 9 is certain enough to act on, and which needs more patients? Who failed with the current app, and what does that suggest for the accessibility audit of Week 10?

## 7. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE9" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-09/"
               "NB09_usability_testing.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The methods of this week apply to any test of two designs. Each app below has the results of a small test of its current design and of a redesign: Completions and the times of the completed tasks.

The results are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "banking"
# clinic, banking, e_government, food_delivery, course_registration
APPS = {
    "clinic": ("book an appointment", (8, 13), (11, 13),
               [58, 66, 61, 74, 83, 97, 121, 140],
               [36, 41, 39, 44, 47, 52, 49, 58, 63, 71, 82]),
    "banking": ("send money", (14, 20), (18, 20),
                [42, 55, 47, 61, 73, 52, 66, 58, 49, 88, 70, 51, 95, 60],
                [35, 41, 38, 46, 44, 52, 39, 48, 57, 43, 50, 37, 61, 45,
                 40, 54, 47, 42]),
    "e_government": ("upload a document", (9, 16), (13, 16),
                     [118, 96, 143, 171, 125, 102, 188, 134, 150],
                     [84, 97, 75, 110, 92, 121, 88, 79, 103, 95, 131, 86, 99]),
    "food_delivery": ("reorder a meal", (17, 20), (19, 20),
                      [28, 35, 31, 26, 40, 33, 29, 45, 37, 30, 34, 27, 52,
                       32, 38, 29, 36],
                      [22, 25, 21, 27, 24, 30, 23, 26, 28, 22, 33, 25, 24,
                       29, 21, 26, 31, 23, 27]),
    "course_registration": ("register for a course", (10, 18), (15, 18),
                            [212, 185, 240, 198, 265, 230, 205, 251, 190, 278],
                            [150, 172, 141, 166, 158, 190, 147, 175, 162, 139,
                             181, 155, 168, 144, 186])}

task, (xa, na), (xb, nb), times_a, times_b = APPS[APP]
print(f"{APP}: {task}")
for name, x, n, t in [("current", xa, na, times_a),
                      ("redesign", xb, nb, times_b)]:
    _, low, high = adjusted_wald(x, n)
    gm, glow, ghigh = geo_mean_ci(t)
    print(f"  {name}: {x} of {n} completed ({low:.2f} to {high:.2f}), "
          f"time {gm:.0f} s ({glow:.0f} to {ghigh:.0f} s)")
z, p = n1_test(xa, na, xb, nb)
pt = stats.ttest_ind(np.log(times_a), np.log(times_b), equal_var=False).pvalue
print(f"  completion: p = {p:.3f} | times: p = {pt:.4f}")
print(f"  per design for these rates: {per_group(xa / na, xb / nb)} people")

**Python note.** Each entry holds the task, the completions of both designs as pairs of successes and participants, and the times of the completed tasks in seconds.

**Observe and reflect.** In your app, which difference is clear with these few people, and which would need a larger test? Would an A/B test with real users be possible for this app?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Record the time that five classmates need for one task in your development environment, and report the geometric mean with its interval. |
| Web development | Plan an A/B test of two sign-up pages: Choose the completion rates you expect, compute the visitors per page with `per_group` and write the stopping rule. |
| Mobile development | Test one task of a mobile app with six people, code the sessions with a classmate and compute your kappa. |
| Game development | Measure how many of ten players finish the first level without help, and compare the adjusted Wald interval with the plain one. |
| Embedded systems and the Internet of Things | Time the setup of a smart device with five people and compare the arithmetic mean, the median and the geometric mean. |
| Data science | Simulate A/A tests with 20 looks instead of 10 and plot how the share of false alarms grows. |
| Cybersecurity | Compare how many of 15 people spot a phishing e-mail in two designs of a warning, with the N-1 test and the interval of each design. |
| Health informatics | Recruit eight people for a test of a hospital website by the age groups of its patients, and weight the completion rate by the patient mix. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-09).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>